# Spatial evolutionary markets

Traders on a lattice, each a fundamentalist (blue), a momentum trader (red)
or a noise trader (white). Every day each one compares the strategies' recent profits and
what its eight neighbours do, and may switch: herds form, spread and break up while the price
follows the S&P 500, nudged by the traders' net demand. Brock & Hommes (1998)'s adaptive
belief system on a lattice, after an
[open-source implementation](https://github.com/zakhac/Spatial-Evolutionary-Markets); the
rules run on a knowledge graph with
[skabm](https://github.com/remiadon/skabm). S&P 500 daily closes from
[FRED](https://fred.stlouisfed.org/series/SP500) (S&P Dow Jones Indices).

In [ ]:
# uv run --extra voila voila notebooks   (or open this notebook in Jupyter)
import inspect
import io
from pathlib import Path

import httpx
import ipywidgets as w
import matplotlib

matplotlib.use("Agg")  # figures become PNGs for an Image widget
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import Markdown, display
from maplib import Model
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from skabm import template
from skabm.behaviour import trader
from skabm.player import Player
from skabm.simulation import RDFSimulator
from skabm.space import moore_neighbors
from skabm.sparql import register_math, register_polars_random

CACHE = Path("data") / "sp500.csv"  # delete it to refetch
# one trading day, as the implementation's simulation.py runs it
RULES = [trader.market_day, trader.strategy_demand, trader.impact_price, trader.logit_imitation]
LATTICE = inspect.signature(trader.lattice).parameters  # lattice's cited defaults
# dataviz reference palette: blue and red categorical slots, the light neutral for noise
COLOURS = ["#2a78d6", "#e34948", "#f0efec"]
NAMES = ["Fundamentalist", "Momentum", "Noise"]


def sp500() -> pl.DataFrame:
    """Daily S&P 500 closes since July 2017, the window the reproduced repo uses."""
    if not CACHE.exists():
        CACHE.parent.mkdir(parents=True, exist_ok=True)
        response = httpx.get("https://fred.stlouisfed.org/graph/fredgraph.csv",
                             params={"id": "SP500", "cosd": "2017-07-01"}, timeout=60)
        response.raise_for_status()
        CACHE.write_text(response.text)
    return pl.read_csv(CACHE, null_values=["", "."], try_parse_dates=True).drop_nulls()


closes = sp500()
dates = closes["observation_date"]


def unfold(size: int, n_days: int, seed: int, params: dict):
    """The run as it happens: each day, the lattice's strategies, the price and the fundamental."""
    close = closes["SP500"].cast(pl.Float64)
    traders = trader.lattice(size, seed=seed)
    world = Model()
    world.map(template.market, trader.opening_market(close).with_iri())
    world.map(template.day, trader.trading_days(close).with_iri("market"))
    world.map(template.trader, traders.with_iri("market"))
    world.map(template.links("neighbor"), moore_neighbors(traders, torus=True).with_iri("neighbor"))
    sim = RDFSimulator(rules=RULES, params=params, n_periods=n_days,
                       udfs=(register_polars_random, register_math), random_seed=seed)
    for frame in sim.fit_iter(world):
        market = frame.filter(pl.col("class") == "Market").row(0, named=True)
        people = frame.filter(pl.col("class") == "Trader")
        grid = np.zeros((size, size), dtype=np.int8)
        grid[people["y"].cast(pl.Int64).to_numpy(), people["x"].cast(pl.Int64).to_numpy()] = people["strategy"].to_numpy()
        yield {"day": int(market["t"]), "price": market["price"], "fundamental": market["fundamental"],
               "grid": grid, "shares": np.bincount(grid.ravel(), minlength=3) / grid.size}


def draw(history: list, k: int) -> bytes:
    """Day k: the lattice, the price against the fundamental, and the strategies' shares so far."""
    step, days = history[k], [dates[s["day"]] for s in history[: k + 1]]
    fig = plt.figure(figsize=(11, 6), layout="constrained")
    axes = fig.subplot_mosaic([["grid", "price"], ["grid", "shares"]], width_ratios=[1, 1.25])
    grid = axes["grid"]
    grid.imshow(step["grid"], cmap=ListedColormap(COLOURS), vmin=-0.5, vmax=2.5, interpolation="nearest")
    grid.set_axis_off()
    grid.legend(handles=[Patch(facecolor=c, edgecolor="#b4b2a9", label=n) for c, n in zip(COLOURS, NAMES)],
                loc="upper center", bbox_to_anchor=(0.5, 0), ncol=3, frameon=False)
    price = axes["price"]
    price.plot(days, [s["price"] for s in history[: k + 1]], color="#2a2a28", linewidth=1.5, label="price")
    price.plot(days, [s["fundamental"] for s in history[: k + 1]], color=COLOURS[0], linewidth=1.5,
               linestyle="--", label="fundamental (trend)")
    price.set_title("S&P 500 price, as the traders move it", loc="left", fontsize=10)
    price.legend(frameon=False, fontsize=9)
    shares = axes["shares"]
    for i, name in enumerate(NAMES):
        shares.plot(days, [s["shares"][i] for s in history[: k + 1]], linewidth=2,
                    color=COLOURS[i] if i < 2 else "#8a8984", label=name)
    shares.set_ylim(0, 1)
    shares.set_title("Share of traders by strategy", loc="left", fontsize=10)
    shares.legend(frameon=False, fontsize=9, ncol=3)
    for ax in (price, shares):
        ax.spines[["top", "right"]].set_visible(False)
        ax.grid(axis="y", color="#e6e5e0")
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=80)
    plt.close(fig)
    return buffer.getvalue()


wide = {"description_width": "11em"}
cited = {str(k): v for k, v in trader.PARAMETERS.items()}
beta = w.FloatSlider(cited["choice_intensity"], min=0.0, max=5.0, step=0.1, description="Intensity of choice β", style=wide)
gamma = w.FloatSlider(cited["social_pressure"], min=0.0, max=6.0, step=0.1, description="Social pressure γ", style=wide)
sigma = w.FloatSlider(cited["noise_sd"], min=0.0, max=0.2, step=0.01, description="Noise traders' σ", style=wide)
impact = w.FloatLogSlider(cited["price_impact"], base=10, min=-5, max=-1, step=0.25, description="Price impact λ",
                          readout_format=".1e", style=wide)
size = w.SelectionSlider(options=[30, 50, 80, 100], value=50, description="Lattice side", style=wide)
n_days = w.IntSlider(500, min=50, max=len(closes) - 1, step=50, description="Trading days", style=wide)
seed = w.BoundedIntText(0, min=0, max=999, description="Seed", style=wide)
for control in (beta, gamma, sigma, impact, size, n_days, seed):
    control.layout.width = "100%"
numbers = w.HTML("Set the market up on the left, then <b>Run</b> to watch the strategies fight it out.")
picture = w.Image(format="png", layout=w.Layout(display="none"))  # until a run draws


def play():
    params = {"choice_intensity": beta.value, "social_pressure": gamma.value, "noise_sd": sigma.value,
              "price_impact": impact.value}
    return unfold(size.value, n_days.value, seed.value, params)


def show(history: list, k: int) -> None:
    step = history[k]
    fund, chart, noise = step["shares"]
    numbers.value = (f"<b>{dates[step['day']]}</b> &nbsp;·&nbsp; price <b>{step['price']:,.0f}</b> "
                     f"(fundamental {step['fundamental']:,.0f}) &nbsp;·&nbsp; fundamentalists <b>{fund:.0%}</b>, "
                     f"momentum <b>{chart:.0%}</b>, noise <b>{noise:.0%}</b>")
    picture.value = draw(history, k)
    picture.layout.display = None


player = Player(play, show, pace=0.0, unit="day")
w.HBox([
    w.VBox([beta, gamma, sigma, impact, size, n_days, seed, player.pace, player.button], layout=w.Layout(width="380px")),
    w.VBox([numbers, player.slider, picture, player.errors]),
])

In [ ]:
source = inspect.getsource(trader)
written = source[source.index("# What each strategy asks for") : source.index("\n\n\ndef lattice(")]
prose = w.Output()
with prose:
    display(Markdown(f"```python\n{written}\n```"))
w.Accordion([prose], titles=("The rules, as the model writes them",))